## PyTorch 的自动微分引擎，也称为 autograd

### 逻辑回归

In [1]:
import torch
import torch.nn.functional as F

y = torch.tensor([1.0]) # 标签
x1 = torch.tensor([1.1]) # input feature
w1 = torch.tensor([2.2])
b = torch.tensor([0.0]) # bias unit

z = x1 * w1 + b # net input
a = torch.sigmoid(z) # activation & output

loss = F.binary_cross_entropy(a, y)
print(loss)

tensor(0.0852)


In [4]:
import torch
import torch.nn.functional as F

y = torch.tensor([1.0]) # 标签
x1 = torch.tensor([1.1]) # input feature
w1 = torch.tensor([2.2])
b = torch.tensor([0.0]) # bias unit

z = x1 * w1 + b # net input
torch.sigmoid(z) # activation & output


tensor([0.9183])

In [6]:
import torch
import torch.nn.functional as F

class LogisticRegression(torch.nn.Module):
    def __init__(self):
        super(LogisticRegression, self).__init__()
        self.linear = torch.nn.Linear(1, 1)

    def forward(self, x):
        y_pred = torch.sigmoid(self.linear(x))
        return y_pred
    

y = torch.tensor([1.0]) # 标签
x1 = torch.tensor([1.1]) # input feature
w1 = torch.tensor([2.2])
b = torch.tensor([0.0]) # bias unit
    
model = LogisticRegression()
loss_fn = F.binary_cross_entropy(model(x1), y)
print(loss_fn)

tensor(0.3607, grad_fn=<BinaryCrossEntropyBackward0>)


## 导数在autograd中的应用详解

### 1. 什么是autograd？

autograd是PyTorch的自动微分引擎，它能够自动计算梯度（导数）。在深度学习中，我们需要计算损失函数对模型参数的导数，以便使用梯度下降等优化算法来更新参数。

### 2. 计算图（Computational Graph）

PyTorch使用计算图来跟踪所有的操作，每个张量都有一个`grad_fn`属性，指向创建它的操作。这形成了一个有向无环图（DAG），记录了从输入到输出的计算路径。

### 3. 梯度计算的基本原理

当我们调用`loss.backward()`时，autograd会：
1. 从损失函数开始，沿着计算图反向传播
2. 使用链式法则计算每个参数的梯度
3. 将梯度存储在参数的`.grad`属性中


您上传的图片和这个问题非常好，它精准地抓住了深度学习（乃至整个机器学习）的核心机制。链式法则和求导虽然不是直接在您写的代码中可见，但它们是PyTorch框架在后台**自动执行反向传播（Backpropagation）时赖以工作的根本数学原理**。

我们可以把您的代码拆解成一个计算图，链式法则的作用就一目了然了。

### 计算图与链式法则

您的代码定义了一个简单的计算过程，这个过程可以看作数据流经一个由函数组成的网络：

`x1, w1, b` → **乘法与加法** → `z` → **Sigmoid函数** → `a` → **BCE损失函数** → `loss`

为了让模型学习（即更新参数 `w1` 和 `b` 以减少 `loss`），我们需要知道：
*   `loss` 对 `w1` 的导数是多少？(`∂loss/∂w1`)
*   `loss` 对 `b` 的导数是多少？(`∂loss/∂b`)

这些导数（梯度）告诉我们，微调 `w1` 或 `b` 会对最终损失 `loss` 产生多大的影响。

**链式法则正是用来计算这些复杂导数（梯度）的规则！** 它允许我们将对 `w1` 的求导过程，分解为沿着计算路径的各个简单函数求导的乘积。

---

### 一个具体的推导示例

让我们用链式法则来手动推导一下 `loss` 如何依赖于 `w1`。

我们的目标是求 `∂loss/∂w1`。

根据计算图，`loss` 依赖于 `a`, `a` 依赖于 `z`, `z` 依赖于 `w1`。因此，应用链式法则：

\[
\frac{\partial \text{loss}}{\partial w_1} = \frac{\partial \text{loss}}{\partial a} \cdot \frac{\partial a}{\partial z} \cdot \frac{\partial z}{\partial w_1}
\]

现在，我们逐项分析，这正好对应了您代码中的三个关键步骤：

1.  **`∂z/∂w1` (梯度来自: `z = x1 * w1 + b`)**
    *   这是最直接的一步。对于 `z = x1 * w1 + b`，求 `z` 关于 `w1` 的偏导数，结果是 `x1`。
    *   **计算值**：`x1` (在你的例子中是 `1.1`)

2.  **`∂a/∂z` (梯度来自: `a = torch.sigmoid(z)`)**
    *   Sigmoid函数的导数是一个已知公式：`∂a/∂z = a * (1 - a)`。
    *   **计算值**：这取决于前向传播计算出的 `a` 值。假设 `a` 算出来是 `0.9`，那么这里就是 `0.9 * (1 - 0.9) = 0.09`。

3.  **`∂loss/∂a` (梯度来自: `loss = F.binary_cross_entropy(a, y)`)**
    *   二元交叉熵损失函数(BCE)关于预测值 `a` 的导数也有一个公式。对于单个样本，它可以简化为：`(a - y) / (a * (1 - a))`。
    *   **计算值**：这取决于 `a` 和真实标签 `y`。假设 `y=1.0`, `a=0.9`，那么这里就是 `(0.9 - 1.0) / (0.9 * (1 - 0.9)) ≈ -1.11`。

**最后，根据链式法则，将这三部分乘起来：**
\[
\frac{\partial \text{loss}}{\partial w_1} = (-1.11) \cdot (0.09) \cdot (1.1) \approx -0.11
\]

这个结果 `-0.11` 就是 `loss` 对于 `w1` 的梯度。PyTorch的优化器（如SGD或Adam）就会用这个值来更新 `w1`：`w1 = w1 - learning_rate * (-0.11)`，从而试图降低损失。

### PyTorch 的自动化：Autograd

您不需要手动进行上述推导。当您使用 `torch.tensor`（并且默认 `requires_grad=True`）并进行运算时，PyTorch 的 **Autograd（自动求导）** 系统会自动：

1.  **跟踪计算图**：记录所有从具有`requires_grad=True`的张量出发的运算步骤。
2.  **在 `.backward()` 时应用链式法则**：当您在最终的 `loss` 张量上调用 `.backward()` 方法时，PyTorch 会从 `loss` 开始，**沿着计算图反向传播**，利用链式法则自动计算所有参与运算的叶节点（您的参数 `w1`, `b`）的梯度，并将计算结果（如 `-0.11`）填充到 `w1.grad` 属性中。

### 总结

| 您的代码（前向传播） | PyTorch Autograd（反向传播） | 数学原理 |
| :--- | :--- | :--- |
| `z = x1 * w1 + b` | 计算 `∂z/∂w1`, `∂z/∂b` | 基础求导 |
| `a = torch.sigmoid(z)` | 计算 `∂a/∂z` | Sigmoid导数公式 |
| `loss = F.binary_cross_entropy(a, y)` | 计算 `∂loss/∂a` | BCE损失导数公式 |
| **无直接代码** | **自动计算 `∂loss/∂w1 = (∂loss/∂a) * (∂a/∂z) * (∂z/∂w1)`** | **链式法则** |

所以，**链式法则和求导是连接您的前向传播代码（计算损失）与模型学习能力（通过梯度更新参数）之间的桥梁**。您负责定义前向计算（如何从输入得到损失），PyTorch 则利用微积分中的链式法则，自动完成反向计算（如何从损失得到参数的更新方向）。

In [14]:
import torch.nn.functional as F
from torch.autograd import grad

y = torch.tensor([1.0], requires_grad=True)
x1 = torch.tensor([1.1], requires_grad=True)
w1 = torch.tensor([0.5], requires_grad=True)
b = torch.tensor([0.3], requires_grad=True)

z = x1 * w1 + b
a = torch.sigmoid(z)

loss = F.binary_cross_entropy(a, y)

print(z)

grad_L_wl = grad(loss, w1, create_graph=True)[0]
print(grad_L_wl)
grad_L_b = grad(loss, b, create_graph=True)[0]
print(grad_L_b)

tensor([0.8500], grad_fn=<AddBackward0>)
tensor([-0.3294], grad_fn=<MulBackward0>)
tensor([-0.2994], grad_fn=<SigmoidBackwardBackward0>)


In [15]:
loss.backward()
print(w1.grad)
print(b.grad)

tensor([-0.3294])
tensor([-0.2994])


In [18]:
a = torch.tensor([1.0]) 
print(a.requires_grad)

False
